# JiT-S2-VMamba — Derf instead of RMSNorm

Every block normalises its input twice (before SS2D and before the FFN) and the head once more, all with RMSNorm, which computes a per-token statistic. This notebook swaps those 25 norms for **Dynamic erf** (Chen, Lu, Zhu, Sun, Liu, *Stronger Normalization-Free Transformers*, CVPR 2026), a point-wise function with no statistics at all:

$$\mathrm{Derf}(x) = \gamma \odot \mathrm{erf}(\alpha x + s) + \beta$$

$\alpha$, $s$ are learnable **scalars** per layer (init 0.5, 0), $\gamma$, $\beta$ per channel (init 1, 0), the paper's defaults. These are the paper's positions (pre-attention → pre-mixer, pre-FFN, final), and how its code swaps the RMSNorms of Caduceus, its one Mamba model. SS2D's post-scan LayerNorm (`out_norm`) is not a pre-norm, was never tested in the paper, and stays.

| norm | per swapped norm | model |
|---|---|---|
| RMSNorm (baseline) | γ | 31,031,616 |
| **Derf (this one)** | γ, β, α, s (+386) | **31,041,266** (+25 × 386) |

The swap draws no random numbers, so at seed 42 every other weight is the baseline's exactly. adaLN-Zero is kept and every conditioning arm is **off**, so the only variable vs the baseline is the norm. (The paper's DiT recipe removes the zero-init for point-wise norms, but its released code keeps it on DiT-B's blocks; that would be a second change.) In the paper Derf beats LN by 1–2 FID on latent DiT-B/L/XL; pixel-space JiT and SS2D are untested, so the gain is a question, not a given. It reports no seeds, so a second seed for this and the baseline is what makes a gap meaningful.

---

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. Derf (src/primitives.py: Derf, convert_rmsnorm_to_derf),
# the norm flag in vmamba.py, the build_model wiring in BOTH run_experiment.py
# and evaluate.py, the Derf config, and the test are all on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the norm wiring is present in both entry points
!grep -q '"derf_alpha_init"' scripts/run_experiment.py && grep -q '"derf_alpha_init"' scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify Derf  *(first session only; ~1-2 min)*

In [ ]:
# === Verify Derf (CPU proofs, ~1 min; first session only -- comment out on resume).
# Reference scan, no GPU: norm=rmsnorm bit-identical to HEAD for all four models
# (init + perturbed output); Derf == a verbatim copy of the official Dynamic_erf
# (fp32 max|diff| 0); exactly norm1 / norm2 / norm_final swapped (25 here) with
# every baseline weight kept; grads reach alpha / shift / weight / bias; no
# weight decay on them; train/eval build_model agree on the Derf config.
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If it FAILS, STOP -- do not train a model that is not the one reported.
!PYTHONPATH=. python tests/test_derf_cpu.py

In [ ]:
# === GPU check (~1 min; first session only). Derf under the training's bf16
# autocast, then the baseline vs the Derf model at the training batch:
# 1) Derf on the GPU (fp32 and bf16 autocast) vs a float64 CPU copy of the formula;
# 2) both models built by run_experiment.py's build_model, fwd+bwd under bf16
#    autocast with finite loss/grads, it/s and peak memory -- note the it/s for
#    the complexity table. If either FAILS, STOP.
import ast, os, sys, time, torch, yaml
sys.path.insert(0, os.getcwd())                     # the repo (cd'ed into above)
from src.primitives import Derf
dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev == "cpu":
    print("WARNING: no GPU visible -- running the check on CPU (tiny batch)")

torch.manual_seed(0)
d = Derf(384).to(dev)
with torch.no_grad():
    d.alpha.fill_(0.7); d.shift.fill_(-0.2); d.weight.normal_(); d.bias.normal_()
x = 3 * torch.randn(8, 64, 384, device=dev)
p = {k: v.detach().double().cpu() for k, v in d.state_dict().items()}
ref = p["weight"] * torch.erf(p["alpha"] * x.double().cpu() + p["shift"]) + p["bias"]
with torch.no_grad():
    e32 = (d(x).double().cpu() - ref).abs().max().item()
    with torch.autocast(dev, dtype=torch.bfloat16):
        y16 = d(x.bfloat16())
    e16 = ((y16.double().cpu() - ref).abs() / (ref.abs() + 1)).max().item()
assert y16.dtype == torch.bfloat16 and e32 < 1e-5 and e16 < 2e-2, (e32, e16)
print(f"1) Derf on {dev}: fp32 max|diff| vs float64 = {e32:.1e}, "
      f"bf16 max rel diff = {e16:.1e}  OK")

src = open("scripts/run_experiment.py", encoding="utf-8").read()
fn = next(n for n in ast.parse(src).body
          if isinstance(n, ast.FunctionDef) and n.name == "build_model")
ns = {}
exec(compile(ast.Module([fn], []), "scripts/run_experiment.py", "exec"), ns)
B = 128 if dev == "cuda" else 2
xb = torch.randn(B, 3, 64, 64, device=dev)
tb = torch.rand(B, device=dev)
yb = torch.randint(0, 200, (B,), device=dev)
stats = {}
for name in ("baseline", "derf"):
    cfg = yaml.safe_load(open(f"configs/tiny_imagenet/jit-s2-vmamba-{name}.yaml"))
    torch.manual_seed(0)
    model = ns["build_model"](cfg).to(dev).train()
    n_derf = sum(isinstance(m, Derf) for m in model.modules())
    if dev == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    def step():
        model.zero_grad(set_to_none=True)
        with torch.autocast(dev, dtype=torch.bfloat16):
            out = model(xb, tb, yb)
        loss = (out.float() - xb).pow(2).mean()
        loss.backward()
        return loss
    for _ in range(3):
        loss = step()
    n_it = 20 if dev == "cuda" else 1
    if dev == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_it):
        loss = step()
    if dev == "cuda": torch.cuda.synchronize()
    its = n_it / (time.time() - t0)
    assert torch.isfinite(loss) and all(torch.isfinite(q.grad).all()
                                        for q in model.parameters() if q.grad is not None)
    mem = torch.cuda.max_memory_allocated() / 2**30 if dev == "cuda" else float("nan")
    stats[name] = (its, mem)
    print(f"2) {name:8s}: {sum(q.numel() for q in model.parameters()):,} params, "
          f"{n_derf} Derf, {its:.2f} it/s, peak {mem:.2f} GiB (batch {B}, bf16)")
    del model
assert stats["derf"][0] > 0
print(f"   Derf / baseline speed: {stats['derf'][0] / stats['baseline'][0]:.2f}x  OK")

## 4. Tiny-ImageNet  *(attached dataset, else download to /tmp)*

In [ ]:
# Tiny-ImageNet-200 -> /tmp (EPHEMERAL), ImageFolder layout for train/ and val/.
# Sources, in order: an attached Kaggle dataset containing tiny-imagenet-200/,
# then Stanford (https, http), then the SJTU mirror of the identical zip.
import os, zipfile, glob, shutil, subprocess, hashlib
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
MD5      = "90528d7ca1a48142e341f4ef8d21d0de"
SOURCES  = ["https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "http://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "https://scidata.sjtu.edu.cn/records/swpcp-01e87/files/tiny-imagenet-200.zip?download=1"]

def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    attached = [os.path.dirname(p) for p in
                glob.glob("/kaggle/input/**/tiny-imagenet-200/train", recursive=True)]
    if attached:
        print("copying attached dataset:", attached[0])
        shutil.copytree(attached[0], DATA_DIR)
    else:
        if os.path.exists(ZIP) and (not zipfile.is_zipfile(ZIP) or _md5(ZIP) != MD5):
            print(f"removing broken download ({os.path.getsize(ZIP)} bytes)")
            os.remove(ZIP)
        for src in SOURCES:
            if os.path.exists(ZIP):
                break
            print("downloading", src, "(~248MB)...")
            r = subprocess.run(["wget", "-q", "--tries=3", "--timeout=60", "-O", ZIP, src])
            ok = r.returncode == 0 and zipfile.is_zipfile(ZIP) and _md5(ZIP) == MD5
            if not ok:
                size = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
                print(f"  failed (wget exit {r.returncode}, {size} bytes)")
                if os.path.exists(ZIP):
                    os.remove(ZIP)
        assert os.path.exists(ZIP), ("Tiny-ImageNet download failed from every source -- "
                                     "attach a Kaggle tiny-imagenet dataset (Add Input) and rerun this cell")
        print("extracting...")
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

n_tr, n_val = len(glob.glob(tr + "/*")), len(glob.glob(val + "/*"))
print("train classes:", n_tr, "| val classes:", n_val, "| DATA_DIR:", DATA_DIR)
assert n_tr == 200 and n_val == 200, "unexpected class count -- dataset incomplete"

## 5. Settings  *(edits the cloned config; finds the checkpoint to resume)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
NORM       = "derf"       # this notebook: Derf in place of norm1 / norm2 / norm_final
ALPHA_INIT = 0.5          # the paper's default (1.0 for its DiT-XL)
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = f"configs/tiny_imagenet/jit-s2-vmamba-{NORM}.yaml"
RUN     = f"exp_norm_{NORM}"
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). Every conditioning arm stays OFF and the FFN stays SwiGLU (adaLN-Zero
# baseline scaffold), so the norm is the only variable vs the baseline.
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "norm": NORM, "derf_alpha_init": ALPHA_INIT,
                 "ffn": "swiglu", "in_context_len": 0, "state_init": "none",
                 "ssc": "none", "adaln_cond": "full"})

# ── Resume across sessions AND accounts (automatic) ────────────────────
# Looks for THIS model's newest checkpoint in: this session's OUT_DIR, any
# attached notebook output (.../checkpoint-last.pt), and any attached DATASET
# holding an uploaded checkpoint (Kaggle unpacks an uploaded .pt into a folder
# with data.pkl / data/ / byteorder / version -- it is zipped back here).
# A checkpoint is used only if its weights match this config's model exactly
# (same names and shapes), so another run's checkpoint is never picked up.
# The newest epoch wins. To force one, set RESUME_CKPT_FORCE to its path.
RESUME_CKPT_FORCE = None
import os, glob, zipfile, torch, yaml
from scripts.evaluate import build_model
_ref = {k: tuple(v.shape) for k, v in build_model(yaml.safe_load(open(CONFIG))).state_dict().items()}

def _rebuild(folder):
    """Zip an unpacked checkpoint folder back into a loadable .pt in /tmp."""
    dst = os.path.join("/tmp/resume", folder.strip("/").replace("/", "__") + ".pt")
    if not os.path.exists(dst):
        os.makedirs("/tmp/resume", exist_ok=True)
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
            for root, _, files in os.walk(folder):
                for f in files:
                    full = os.path.join(root, f)
                    z.write(full, os.path.join("checkpoint", os.path.relpath(full, folder)))
    return dst

cands = [os.path.join(OUT_DIR, "checkpoint-last.pt")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoint-last.pt", recursive=True))
cands += [_rebuild(os.path.dirname(p))
          for p in sorted(glob.glob("/kaggle/input/**/data.pkl", recursive=True))]
best = None
print("checkpoints found:")
for c in ([RESUME_CKPT_FORCE] if RESUME_CKPT_FORCE else cands):
    if not (c and os.path.exists(c)):
        continue
    try:
        p = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  unreadable ({type(e).__name__}): {c}")
        continue
    match = {k: tuple(v.shape) for k, v in p.get("model", {}).items()} == _ref
    print(f"  {'THIS model ' if match else 'other model'}  epoch {p.get('epoch')}  {c}")
    if match and (best is None or p["epoch"] > best[0]):
        best = (p["epoch"], c)
    del p
RESUME_CKPT = best[1] if best else None

print("config :", CONFIG, "| norm:", NORM, "| alpha init:", ALPHA_INIT,
      "| arms: all off (adaLN-Zero baseline)")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR)
print("resume :", f"{RESUME_CKPT} (last finished epoch {best[0]})" if best else "(fresh start)")

## 6. Train  *(Save Version before 12h; next session it resumes by itself)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- **Resuming, also on another Kaggle account:** attach the previous session's notebook output, or upload its `checkpoint-last.pt` as a dataset and attach that. The settings cell finds it, keeps only checkpoints of this exact model (Derf adds `bias` / `alpha` / `shift` to each of the 25 swapped norms, so a baseline RMSNorm checkpoint never matches), and resumes from the newest epoch. Check its `resume :` line before training.
- **Data:** the Tiny-ImageNet cell uses an attached dataset first, then Stanford, then the SJTU mirror of the identical zip (MD5-checked). To skip downloads for good, upload `tiny-imagenet-200.zip` once as your own Kaggle dataset and attach it.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the baseline and every other run, so the only variable is the norm.
- +9,650 params (+0.03% of the model): a gap is the norm's behaviour, not capacity. Derf is 5 unfused element-wise ops vs RMSNorm's ~6 (also unfused here), so the speed should be close -- record the it/s and peak memory printed by the GPU check for the complexity table.
- After training, the learned α / s per layer are worth a look (`model.blocks[i].norm1.alpha`, `.shift`): they show how hard each layer saturates its input, which is the paper's explanation for why erf beats tanh.
- `norm` is orthogonal to the conditioning arms and the FFN: to combine Derf with the best arm, add `norm: derf` to that arm's config -- no code change. It also works for `vmamba_sasf`, `spatial_mamba` and `vmamba2`.
- Paper details that matter here: α init 0.5 (its DiT-XL used 1.0 -- set `ALPHA_INIT`); its DiT runs drop adaLN-Zero for point-wise norms but keep it on DiT-B's blocks in the released code; it tunes the LR per model over {1e-4, 2e-4, 4e-4}. This run changes the norm only.